# Segment your dataset with Segment Anything 3 (SAM3)

---


This notebook focuses on running SAM3 inference on a set of images to produce a mAP score for the purpose of benchmarking.

_Note: This is an adaptation of Roboflow's notebook on SAM 3 (Segment Anything Model 3) [available here](https://github.com/roboflow/notebooks/blob/main/notebooks/how-to-segment-images-with-segment-anything-3.ipynb)._


# SAM 3 Setup 

## Environment setup

### Configure your API keys

To pull Segment Anything 3 weights, you need a HuggingFace Access Token with approved access to the SAM 3 checkpoints.

- Request access to the SAM 3 checkpoints on the official Hugging Face [repo](https://huggingface.co/facebook/sam3).
- Open your HuggingFace Settings page. Click Access Tokens then New Token to generate a new token.
- In Colab, go to the left pane and click on Secrets (🔑). Store your HuggingFace Access Token under the name `HF_TOKEN`.








In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

### Check GPU availability

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `T4 GPU`, and then click `Save`.

In [ ]:
!nvidia-smi

In [ ]:
import torch
import torchvision

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA is available:", torch.cuda.is_available())

### Install SAM 3 and extra dependencies

In [ ]:
!git clone https://github.com/facebookresearch/sam3.git
%cd sam3
!pip install -e ".[notebooks]"
%cd /content

In [ ]:
!pip install -q supervision jupyter_bbox_widget

## Load SAM3 Image Predictor

On Ampere GPUs (compute capability ≥ 8), we enable TensorFloat-32 (TF32) for matrix multiplications and convolutions. This allows PyTorch to use tensor cores to accelerate FP32 computations while maintaining similar numerical accuracy.

In [ ]:
import torch

torch.autocast(device_type="cuda", dtype=torch.bfloat16).__enter__()

if torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

In [ ]:
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

model = build_sam3_image_model()
processor = Sam3Processor(model, confidence_threshold=0.3)

## Few utils to parse and visualize the result

In [ ]:
import supervision as sv

def from_sam(sam_result: dict) -> sv.Detections:
    xyxy = sam_result["boxes"].to(torch.float32).cpu().numpy()
    confidence = sam_result["scores"].to(torch.float32).cpu().numpy()

    mask = sam_result["masks"].to(torch.bool)
    mask = mask.reshape(mask.shape[0], mask.shape[2], mask.shape[3]).cpu().numpy()

    return sv.Detections(
        xyxy=xyxy,
        confidence=confidence,
        mask=mask
    )

In [ ]:
import supervision as sv
from PIL import Image
from typing import Optional


COLOR = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
    "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])


def annotate(image: Image.Image, detections: sv.Detections, label: Optional[str] = None) -> Image.Image:
    text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)

    mask_annotator = sv.MaskAnnotator(
        color=COLOR,
        color_lookup=sv.ColorLookup.INDEX,
        opacity=0.6
    )
    box_annotator = sv.BoxAnnotator(
        color=COLOR,
        color_lookup=sv.ColorLookup.INDEX,
        thickness=1
    )
    label_annotator = sv.LabelAnnotator(
        color=COLOR,
        color_lookup=sv.ColorLookup.INDEX,
        text_scale=0.4,
        text_padding=5,
        text_color=sv.Color.BLACK,
        text_thickness=1
    )

    annotated_image = image.copy()
    annotated_image = mask_annotator.annotate(annotated_image, detections)
    annotated_image = box_annotator.annotate(annotated_image, detections)

    if label:
        labels = [
            f"{label} {confidence:.2f}"
            for confidence in detections.confidence
        ]
        annotated_image = label_annotator.annotate(annotated_image, detections, labels)

    return annotated_image

# Run inference & get labels

In [ ]:
# Optionally setup Google drive in Colab (do i need this?)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import kagglehub
from pathlib import Path
from PIL import Image
from IPython.display import display
import supervision as sv
import numpy as np

def download_dataset(dataset_handle):
    """Download the Kaggle dataset."""
    try:
        path = kagglehub.dataset_download(dataset_handle)
        print(f"Downloaded dataset to: {path}")
        return path
    except Exception as e:
        print(f"Failed to download dataset: {e}")
        raise


In [ ]:
# download dataset
dataset_handle = 'nmata010/overhead-potholes-test-set-v1'
dataset_path = download_dataset(dataset_handle)

# target test images of above dataset
folder_path = Path(f"{dataset_path}/test/images")

# target results path
val_results_path = Path('/content/drive/MyDrive/Cvis-Potholes/Benchmarks')
name = '12102025_SAM3' # new folder name created in save path
save_dir = val_results_path / name
save_dir.mkdir(parents=True, exist_ok=True) # create the save loc
object_count_filename = f"{name}.csv" # use the benchmark name for csv
object_count_file = save_dir / object_count_filename # create the object count file

In [ ]:
# run inference on all test images
for file in folder_path.iterdir():
  if file.is_file():


    PROMPT = "pothole" # the object we're detecting
    IMAGE_PATH = file.resolve() # the current image file


    image = Image.open(IMAGE_PATH).convert("RGB")
    inference_state = processor.set_image(image)
    inference_state = processor.set_text_prompt(state=inference_state, prompt=PROMPT)

    detections = from_sam(sam_result=inference_state)
    detections = detections[detections.confidence > 0.5]

    annotated_img = annotate(image, detections, label=PROMPT) # gen the annotated image

    save_path = save_dir / file.name # create a save path for the annotated image
    annotated_img.save(save_path) # save the annotated image to the save path

    results_filename = f"{file.stem}.txt" # use the image name as the results filename
    results_file = save_dir / results_filename # create the results file

    with open(results_file, 'w') as f: # open the results file
      width, height = image.size # get the current image size

      for mask in detections.mask:
        polygons = sv.mask_to_polygons(mask)
        for polygon in polygons:
          normalized_polygon = polygon / np.array([width, height])
          flat_polygon = normalized_polygon.flatten()
          coordinate_str = " ".join(map(str, flat_polygon))
          print(f"0 {coordinate_str}", file=f, flush=True)

    with open(object_count_file, 'a') as f: # open the object count file & append
      f.write(f"{file.stem}, {len(detections)}\n")
      print(f"There are {len(detections)} {PROMPT} objects detected in the image: {file.name}")

  # display(annotated_img) # uncomment this to show the image in colab console





